In [1]:
import pandas as pd

df = pd.read_csv(r"C:\Users\DELL\Downloads\infosys-carlease-contract-ai-group2-main\infosys-carlease-contract-ai-group2\data\sample_emails.csv")
df.head()


,id,sender,subject,body,priority,label
0,1,hr@company.com,Offer Letter - Please Sign,"Dear Candidate, Congratulations! Please sign a...",high,important
1,2,noreply@shopping.com,Big Billion Sale is Live,Flat 70% OFF on electronics. Limited period of...,low,promotion
2,3,alerts@mybank.com,Unusual Login Attempt,We detected a login attempt to your account fr...,high,security
3,4,newsletter@blog.com,Weekly Tech Newsletter,"In this week's edition, learn about AI agents,...",low,newsletter
4,5,friend123@gmail.com,Coffee this weekend?,"Hey, long time no see! Are you free this weeke...",medium,personal


In [2]:
df.info()
df.describe()
df.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        8 non-null      int64 
 1   sender    8 non-null      object
 2   subject   8 non-null      object
 3   body      8 non-null      object
 4   priority  8 non-null      object
 5   label     8 non-null      object
dtypes: int64(1), object(5)
memory usage: 516.0+ bytes


,id,sender,subject,body,priority,label
0,1,hr@company.com,Offer Letter - Please Sign,"Dear Candidate, Congratulations! Please sign a...",high,important
1,2,noreply@shopping.com,Big Billion Sale is Live,Flat 70% OFF on electronics. Limited period of...,low,promotion
3,4,newsletter@blog.com,Weekly Tech Newsletter,"In this week's edition, learn about AI agents,...",low,newsletter
7,8,training@lms.com,Reminder: Complete Mandatory Training,Your mandatory security awareness training is ...,medium,work
2,3,alerts@mybank.com,Unusual Login Attempt,We detected a login attempt to your account fr...,high,security


In [5]:
import pandas as pd
import re

def parse_contract(text):
    result = {}

    # Extract APR
    apr_match = re.search(r'APR (\d+\.?\d*)%', text)
    result['apr_extracted'] = float(apr_match.group(1)) if apr_match else None

    # Extract term
    term_match = re.search(r'(\d+) months', text)
    result['term_extracted'] = int(term_match.group(1)) if term_match else None

    # Extract payment
    payment_match = re.search(r'monthly payment \$?(\d+)', text)
    result['monthly_payment_extracted'] = int(payment_match.group(1)) if payment_match else None

    # Extract penalty
    penalty_match = re.search(r'(Late fee \$\d+|Early termination fee \$\d+|None)', text)
    result['penalty_extracted'] = penalty_match.group(1) if penalty_match else None

    return result

parsed = df['body'].apply(parse_contract)
parsed_df = pd.DataFrame(parsed.tolist())
parsed_df.head()


,apr_extracted,term_extracted,monthly_payment_extracted,penalty_extracted
0,None,None,None,None
1,None,None,None,None
2,None,None,None,None
3,None,None,None,None
4,None,None,None,None


In [7]:
# STEP 1: Parse contract text
parsed = df['body'].apply(parse_contract)
parsed_df = pd.DataFrame(parsed.tolist())

# STEP 2: Merge parsed results into df
df = pd.concat([df, parsed_df], axis=1)

# STEP 3: Risk rule
def risk_rule(row):
    if row['apr_extracted'] and row['apr_extracted'] > 10:
        return "HIGH"
    if row['monthly_payment_extracted'] and row['monthly_payment_extracted'] > 900:
        return "MEDIUM"
    return "LOW"

# STEP 4: Apply rule
df['rule_based_risk'] = df.apply(risk_rule, axis=1)

# STEP 5: Show result
df[['body', 'rule_based_risk']].head()


,body,rule_based_risk
0,"Dear Candidate, Congratulations! Please sign a...",LOW
1,Flat 70% OFF on electronics. Limited period of...,LOW
2,We detected a login attempt to your account fr...,LOW
3,"In this week's edition, learn about AI agents,...",LOW
4,"Hey, long time no see! Are you free this weeke...",LOW


In [9]:
final_output = df[['id',
                   'sender',
                   'apr_extracted',
                   'monthly_payment_extracted',
                   'term_extracted',
                   'rule_based_risk',
                   'label']]


In [10]:
final_output.to_csv("milestone1_your name.csv", index=False)